In [ ]:
"""Synthetic hotel-booking dataset generator.

Tables: customers, properties, room_types, rooms, promotions, bookings,
payments, departments, employees. All foreign keys are consistent.
"""
from pathlib import Path

import numpy as np
import pandas as pd
from faker import Faker

SEED = 203
rng = np.random.default_rng(SEED)           # one RNG for numpy-side randomness
Faker.seed(SEED)                            # reproducible Faker output
fake = Faker(["en_US", "fr_FR", "de_DE", "es_ES", "pt_PT", "pl_PL", "es_MX"])
fake_us = Faker("en_US")                    # separate instance: phone / city formats

ROOM_TYPES = {  # name: (max_capacity, base_cost)
    "Single": (1, 75),
    "Double": (2, 125),
    "Triple": (3, 210),
    "Suite": (4, 495),
}
DEPARTMENTS = ["Management", "Front Desk", "Housekeeping", "Maintenance",
               "Food & Beverage", "Sales & Marketing", "Finance"]
CAMPAIGNS = ["Black Friday", "Thanksgiving", "New Year", "Weekly", "Monthly"]


# ----------------------------------------------------------------- helpers
def make_ids(prefix: str, n: int) -> list[str]:
    return [f"{prefix}_{i:05d}" for i in range(1, n + 1)]


def random_dates(start, end, size: int) -> pd.DatetimeIndex:
    """Uniform random dates in [start, end]. `start`/`end` can be scalars or arrays."""
    s = np.broadcast_to(np.asarray(pd.to_datetime(start), dtype="datetime64[D]"), (size,))
    e = np.broadcast_to(np.asarray(pd.to_datetime(end), dtype="datetime64[D]"), (size,))
    span = np.maximum((e - s).astype(int), 0)
    offsets = (rng.random(size) * (span + 1)).astype(int)
    return pd.to_datetime(s + offsets.astype("timedelta64[D]"))


def adult_from(birthdates: pd.DatetimeIndex) -> pd.DatetimeIndex:
    return birthdates + pd.to_timedelta(18 * 365, unit="D")


# ------------------------------------------------------------------ tables
def gen_customers(n: int) -> pd.DataFrame:
    birthdate = random_dates("1955-01-01", "2000-01-01", n)
    earliest = np.maximum(adult_from(birthdate), pd.Timestamp("2018-06-22"))
    return pd.DataFrame({
        "customer_id": make_ids("cust", n),
        "first_name": [fake.first_name() for _ in range(n)],
        "last_name": [fake.last_name() for _ in range(n)],
        "birthdate": birthdate.date,
        "email": [fake.unique.email() for _ in range(n)],
        "phone": [fake_us.unique.phone_number() for _ in range(n)],
        "country": [fake.country() for _ in range(n)],
        "created_at": random_dates(earliest, pd.Timestamp.today().normalize(), n).date,
    })


def gen_room_types(n: int) -> pd.DataFrame:
    types = rng.choice(list(ROOM_TYPES), size=n)
    cap = [ROOM_TYPES[t][0] for t in types]
    # small price variation so several rows of the same type are not identical
    cost = [round(ROOM_TYPES[t][1] * rng.uniform(0.9, 1.3), 2) for t in types]
    return pd.DataFrame({
        "room_type_id": make_ids("rtype", n),
        "type": types,
        "max_capacity": cap,
        "base_cost": cost,
    })


def gen_rooms(n: int, property_ids: list[str], room_type_ids: list[str]) -> pd.DataFrame:
    df = pd.DataFrame({
        "room_id": make_ids("room", n),
        "property_id": rng.choice(property_ids, size=n),
        "room_type_id": rng.choice(room_type_ids, size=n),
    })
    # room numbers restart at 1 inside every property
    df["room_number"] = df.groupby("property_id").cumcount() + 1
    return df[["room_id", "room_number", "property_id", "room_type_id"]]


def gen_properties(n: int, rooms: pd.DataFrame, room_types: pd.DataFrame, ids: list[str]) -> pd.DataFrame:
    # capacity is derived from the rooms the property actually has
    cap = (rooms.merge(room_types[["room_type_id", "max_capacity"]], on="room_type_id")
                .groupby("property_id")["max_capacity"].sum())
    return pd.DataFrame({
        "property_id": ids,
        "name": [f"{fake.word().title()} {fake.last_name()}" for _ in range(n)],
        "city": [fake_us.unique.city() for _ in range(n)],
        "capacity": pd.Series(ids).map(cap).fillna(0).astype(int),
        "stars": rng.integers(3, 6, size=n),          # 3..5 inclusive
    })


def gen_promotions(n: int) -> pd.DataFrame:
    pct = rng.integers(5, 31, size=n)
    valid_from = random_dates("2022-05-12", "2025-12-11", n)
    valid_to = valid_from + pd.to_timedelta(rng.integers(2, 22, size=n), unit="D")
    ids = make_ids("promotion", n)
    return pd.DataFrame({
        "promotion_id": ids,
        # id suffix guarantees unique codes
        "promotion_code": [f"{fake.last_name().upper()}{p}_{i[-5:]}" for p, i in zip(pct, ids)],
        "discount_pct": pct,
        "campaign": rng.choice(CAMPAIGNS, size=n),
        "valid_from": valid_from.date,
        "valid_to": valid_to,
    })


def gen_bookings(n: int, customers, rooms, room_types, promotions) -> pd.DataFrame:
    nights = rng.integers(1, 7, size=n)
    check_in = random_dates("2022-09-12", "2025-01-10", n)
    df = pd.DataFrame({
        "booking_id": make_ids("booking", n),
        "customer_id": rng.choice(customers["customer_id"], size=n),
        "room_id": rng.choice(rooms["room_id"], size=n),
        "number_of_nights": nights,
        "check_in_date": check_in,
        "check_out_date": check_in + pd.to_timedelta(nights, unit="D"),
        "status": rng.choice(["Confirmed", "Cancelled"], size=n, p=[0.85, 0.15]),
        "is_full_package": rng.choice(["Yes", "No"], size=n, p=[0.1, 0.9]),
    })

    # promotion: only if valid on the check-in date (~30% of eligible bookings use one)
    promo_id = pd.Series([None] * n, dtype=object) # [None, None, None]
    for p in promotions.itertuples(): 
        in_window = (df["check_in_date"] >= pd.Timestamp(p.valid_from)) & \
                    (df["check_in_date"] <= pd.Timestamp(p.valid_to))
        take = in_window & promo_id.isna() & (rng.random(n) < 0.30) # True Or False
        promo_id[take] = p.promotion_id
    df["promotion_id"] = promo_id

    # total amount = nights * room base cost, +25% full package, minus promo discount
    base = (df[["room_id"]].merge(rooms[["room_id", "room_type_id"]], on="room_id", how="left")
                           .merge(room_types[["room_type_id", "base_cost"]], on="room_type_id", how="left")
                           ["base_cost"].to_numpy())
    pct = df["promotion_id"].map(promotions.set_index("promotion_id")["discount_pct"]).fillna(0).to_numpy()
    package = np.where(df["is_full_package"] == "Yes", 1.25, 1.0)
    df["total_amount"] = np.round(nights * base * package * (1 - pct / 100), 2)
    return df


def gen_payments(n: int, bookings: pd.DataFrame) -> pd.DataFrame:
    paid = bookings[bookings["status"] == "Confirmed"]
    picked = paid.sample(n=n, replace=True, random_state=SEED).reset_index(drop=True)
    # payment happens 1-29 days before check-in, drawn per row
    created = picked["check_in_date"] - pd.to_timedelta(rng.integers(1, 30, size=n), unit="D")
    return pd.DataFrame({
        "payment_id": make_ids("payment", n),
        "booking_id": picked["booking_id"],
        "amount": picked["total_amount"],
        "bank_provider": rng.choice(["mastercard", "visa", "paypal", "stripe", "crypto"], size=n),
        "status": rng.choice(["completed", "rejected"], size=n, p=[0.95, 0.05]),
        "created_at": created,
    })


def gen_departments() -> pd.DataFrame:
    return pd.DataFrame({"department_id": make_ids("dept", len(DEPARTMENTS)), "name": DEPARTMENTS})


def gen_employees(n: int, property_ids: list[str], departments: pd.DataFrame) -> pd.DataFrame:
    if n < len(property_ids):
        raise ValueError("Need at least one employee (the general manager) per property.")
    n_prop = len(property_ids)
    mgmt_dept = departments.loc[departments["name"] == "Management", "department_id"].iloc[0]
    other_depts = departments.loc[departments["name"] != "Management", "department_id"].to_numpy()

    ids = make_ids("emp", n)
    prop = np.concatenate([property_ids, rng.choice(property_ids, size=n - n_prop)])
    gm_of = dict(zip(property_ids, ids[:n_prop]))          # first n_prop employees = GMs

    birth = random_dates("1960-01-01", "2003-01-01", n)
    hire = random_dates(np.maximum(adult_from(birth), pd.Timestamp("2015-01-01")),
                        pd.Timestamp.today().normalize(), n)
    return pd.DataFrame({
        "employee_id": ids,
        "first_name": [fake.first_name() for _ in range(n)],
        "last_name": [fake.last_name() for _ in range(n)],
        "birthdate": birth.date,
        "hire_date": hire.date,
        "property_id": prop,
        "manager_id": [None] * n_prop + [gm_of[p] for p in prop[n_prop:]],
        "department_id": [mgmt_dept] * n_prop + list(rng.choice(other_depts, size=n - n_prop)),
    })


# -------------------------------------------------------------------- main
def generate_data(n_customers, n_properties, n_room_types, n_rooms,
                  n_promotions, n_bookings, n_payments, n_employees) -> dict[str, pd.DataFrame]:
    property_ids = make_ids("property", n_properties)
    room_types = gen_room_types(n_room_types)
    rooms = gen_rooms(n_rooms, property_ids, room_types["room_type_id"].tolist())
    properties = gen_properties(n_properties, rooms, room_types, property_ids)
    customers = gen_customers(n_customers)
    promotions = gen_promotions(n_promotions)
    bookings = gen_bookings(n_bookings, customers, rooms, room_types, promotions)
    payments = gen_payments(n_payments, bookings)
    departments = gen_departments()
    employees = gen_employees(n_employees, property_ids, departments)
    return {
        "customers": customers, "properties": properties, "room_types": room_types,
        "rooms": rooms, "promotions": promotions, "bookings": bookings,
        "payments": payments, "departments": departments, "employees": employees,
    }


if __name__ == "__main__":
    tables = generate_data(
        n_customers=10_000, n_properties=50, n_room_types=20, n_rooms=2_000,
        n_promotions=40, n_bookings=30_000, n_payments=30_000, n_employees=500,
    )
    out = Path("output")
    out.mkdir(exist_ok=True)
    for name, df in tables.items():
        df.to_csv(out / f"{name}.csv", index=False)
        print(f"{name:<12} {len(df):>7} rows")
    print(tables["customers"].sample(10, random_state=SEED))

customers      10000 rows
properties        50 rows
room_types        20 rows
rooms           2000 rows
promotions        40 rows
bookings       30000 rows
payments       30000 rows
departments        7 rows
employees        500 rows
     customer_id first_name last_name   birthdate  \
4633  cust_04634     Ariana   Mínguez  1973-11-16   
3467  cust_03468    António     Jäkel  1962-05-12   
8712  cust_08713    Rolando   Vicente  1961-10-10   
340   cust_00341  Katherine      Trüb  1962-03-17   
702   cust_00703     Xavier    Hervás  1993-05-15   
3793  cust_03794      Gülay  Carneiro  1958-06-22   
755   cust_00756     Dörthe     Moura  1963-11-23   
6993  cust_06994     Julita   Cardoso  1958-09-09   
8208  cust_08209  Élisabeth    Boutin  1974-01-09   
9536  cust_09537     Elodia  Salvador  1958-09-07   

                               email                   phone     country  \
4633            itziar34@example.org       885.709.5028x6709       Pérou   
3467      alexandraverdu@examp